# 04: Preprocessing and Train/Test Split

This notebook creates the shared **80% train / 20% test split** used by every supervised model in the project.
The split is stratified on `Attrition` and uses `random_state=42`, so each model is evaluated on the same employees.

## 1. Imports

In [10]:
import sys
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from src.config import PROJECT_ROOT

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    TRAIN_FEATURED_PATH,
    MODEL_DATA_DIR,
    TARGET,
    ID_COLUMN,
    RANDOM_STATE,
)

TEST_SIZE = 0.20

## 2. Load the Dataset

In [2]:
df = pd.read_csv(TRAIN_FEATURED_PATH)

print("Shape:", df.shape)
print("Attrition rate:", f"{df[TARGET].mean():.1%}")

Shape: (1058, 41)
Attrition rate: 16.9%


## 3. Created the Train/Test Split (80/20)
`EmployeeNumber` is not used as a model feature. It is saved separately so every model can be checked against the same split.

In [3]:
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    stratify=df[TARGET],
    random_state=RANDOM_STATE,
)

train_df = train_df.sort_values(ID_COLUMN).reset_index(drop=True)
test_df = test_df.sort_values(ID_COLUMN).reset_index(drop=True)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print()
print("Train attrition rate:", f"{train_df[TARGET].mean():.1%}")
print("Test attrition rate:", f"{test_df[TARGET].mean():.1%}")

Train rows: 846
Test rows: 212

Train attrition rate: 16.9%
Test attrition rate: 17.0%


In [4]:
split_manifest = pd.concat([
    train_df[[ID_COLUMN, TARGET]].assign(Split="train"),
    test_df[[ID_COLUMN, TARGET]].assign(Split="test"),
], ignore_index=True).sort_values(ID_COLUMN)

split_manifest.to_csv(
    MODEL_DATA_DIR / "split_manifest.csv",
    index=False,
)

split_manifest["Split"].value_counts()

Split
train    846
test     212
Name: count, dtype: int64

## 4. Define Predictors/Features and Target

In [5]:
feature_columns = [
    column for column in df.columns
    if column not in [TARGET, ID_COLUMN]
]

X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df[TARGET].reset_index(drop=True)
y_test = test_df[TARGET].reset_index(drop=True)

categorical_columns = [
    column for column in X_train.columns
    if not pd.api.types.is_numeric_dtype(X_train[column])
]

numeric_columns = [
    column for column in X_train.columns
    if column not in categorical_columns
]

print("Numeric features:", len(numeric_columns))
print("Categorical features:", len(categorical_columns))

Numeric features: 32
Categorical features: 7


## 5. Logistic Regression dataset

Logistic Regression benefits from scaled numeric features. Categorical variables are one-hot encoded.

Missing-value steps are included as safeguards even though the current IBM dataset has no missing values.

In [6]:
logistic_numeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

logistic_categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )),
])

logistic_preprocessor = ColumnTransformer([
    ("num", logistic_numeric, numeric_columns),
    ("cat", logistic_categorical, categorical_columns),
])

X_train_logistic = logistic_preprocessor.fit_transform(X_train)
X_test_logistic = logistic_preprocessor.transform(X_test)

logistic_feature_names = [
    name.split("__", 1)[-1]
    for name in logistic_preprocessor.get_feature_names_out()
]

logistic_train = pd.DataFrame(
    X_train_logistic,
    columns=logistic_feature_names,
)

logistic_test = pd.DataFrame(
    X_test_logistic,
    columns=logistic_feature_names,
)

logistic_train[TARGET] = y_train.values
logistic_test[TARGET] = y_test.values

logistic_train.to_csv(
    MODEL_DATA_DIR / "logistic_regression_train.csv",
    index=False,
)

logistic_test.to_csv(
    MODEL_DATA_DIR / "logistic_regression_test.csv",
    index=False,
)

print("Train shape:", logistic_train.shape)
print("Test shape:", logistic_test.shape)

Train shape: (846, 61)
Test shape: (212, 61)


## 6. DummyClassifier, Random Forest, XGBoost, and LightGBM datasets

These models do not need standardized numeric values.

For consistency, nominal categorical variables are one-hot encoded and numeric variables are left on their original scale.

The same processed matrix works for all four models, but separate files are saved so each later notebook has an obvious model-specific input.

In [7]:
tree_numeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

tree_categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )),
])

tree_preprocessor = ColumnTransformer([
    ("num", tree_numeric, numeric_columns),
    ("cat", tree_categorical, categorical_columns),
])

X_train_tree = tree_preprocessor.fit_transform(X_train)
X_test_tree = tree_preprocessor.transform(X_test)

tree_feature_names = [
    name.split("__", 1)[-1]
    for name in tree_preprocessor.get_feature_names_out()
]

tree_train = pd.DataFrame(
    X_train_tree,
    columns=tree_feature_names,
)

tree_test = pd.DataFrame(
    X_test_tree,
    columns=tree_feature_names,
)

tree_train[TARGET] = y_train.values
tree_test[TARGET] = y_test.values

for model_name in [
    "dummy_classifier",
    "random_forest",
    "xgboost",
    "lightgbm",
]:
    tree_train.to_csv(
        MODEL_DATA_DIR / f"{model_name}_train.csv",
        index=False,
    )
    tree_test.to_csv(
        MODEL_DATA_DIR / f"{model_name}_test.csv",
        index=False,
    )

print("Train shape:", tree_train.shape)
print("Test shape:", tree_test.shape)


Train shape: (846, 61)
Test shape: (212, 61)


## 7. CatBoost dataset

CatBoost can work directly with categorical variables, so one-hot encoding would throw away one of its main advantages.

For CatBoost:

- categorical columns remain as text
- numeric columns remain on their original scale
- no standardization is applied
- `EmployeeNumber` is excluded

In [8]:
catboost_train = train_df[
    feature_columns + [TARGET]
].copy()

catboost_test = test_df[
    feature_columns + [TARGET]
].copy()

numeric_medians = catboost_train[numeric_columns].median()

for column in numeric_columns:
    catboost_train[column] = catboost_train[column].fillna(
        numeric_medians[column]
    )
    catboost_test[column] = catboost_test[column].fillna(
        numeric_medians[column]
    )

for column in categorical_columns:
    catboost_train[column] = (
        catboost_train[column]
        .fillna("Missing")
        .astype(str)
    )
    catboost_test[column] = (
        catboost_test[column]
        .fillna("Missing")
        .astype(str)
    )

catboost_train.to_csv(
    MODEL_DATA_DIR / "catboost_train.csv",
    index=False,
)

catboost_test.to_csv(
    MODEL_DATA_DIR / "catboost_test.csv",
    index=False,
)

print("Train shape:", catboost_train.shape)
print("Test shape:", catboost_test.shape)

Train shape: (846, 40)
Test shape: (212, 40)


## 8. Sanity Check

In [9]:
print("Training target counts:")
print(y_train.value_counts().sort_index())

print("\nTest target counts:")
print(y_test.value_counts().sort_index())

print("\nTrain percentage:", len(train_df) / len(df))
print("Test percentage:", len(test_df) / len(df))

Training target counts:
Attrition
0    703
1    143
Name: count, dtype: int64

Test target counts:
Attrition
0    176
1     36
Name: count, dtype: int64

Train percentage: 0.7996219281663516
Test percentage: 0.2003780718336484
